In [ ]:
from playwright.async_api import async_playwright
import json

url = "https://cellphones.com.vn/do-gia-dung.html"
async with async_playwright() as p:
  browser = await p.chromium.launch_persistent_context(
      headless=False,
      slow_mo=1000,
      args=[
          "--disable-gpu",
          "--disable-dev-shm-usage",
          "--disable-extensions",
          "--disable-background-networking",
          "--disable-sync",
          "--disable-notifications",
          "--disable-default-apps",
          "--no-sandbox",
      ],
      user_data_dir="profile",
  )

  page = await browser.new_page()

  await page.goto(url)
  await page.wait_for_load_state("domcontentloaded")

  category_elements = await page.query_selector_all(".group.flex.h-full.w-auto.cursor-pointer.flex-col.items-center")

  categories = []

  for category in category_elements:
      img = await category.query_selector("img")
      img_src = await img.get_attribute("src") if img else None
      name = await img.get_attribute("alt") if img else None
      categories.append({
          "name": name,
          "img": img_src
      })
  with open("categories.json", "w", encoding="utf-8") as f:
      json.dump(categories, f, ensure_ascii=False, indent=4)

In [ ]:
import os 
import requests
import unicodedata
import re

def remove_vietnamese_tones(text):
    text = unicodedata.normalize('NFD', text)
    text = re.sub(r'[\u0300-\u036f]', '', text)
    text = text.replace('đ', 'd').replace('Đ', 'D')
    return text
imgs = []

os.makedirs("categories", exist_ok=True)
with open("categories.json", "r", encoding="utf-8") as f:
    categories = json.load(f)
    for category in categories:
        imgs.append({"url": category["img"], "name": category["name"].lower().replace(" ", "_")})
for img in imgs:
    response = requests.get(img["url"])
    if response.status_code == 200:
        filename = remove_vietnamese_tones(img["name"])
        filename += ".png"
        print(f"Downloading {filename}...")
        with open(os.path.join("categories", filename), "wb") as f:
            f.write(response.content)
print("Images downloaded successfully.")

Images downloaded successfully.


In [15]:
with open("categories.json", "w", encoding="utf-8") as f:
  for category in categories:
    category["url"] = remove_vietnamese_tones(category["name"].lower().replace(" ", "_"))
    category.pop("img", None)
  json.dump(categories, f, ensure_ascii=False, indent=4)